# 3DICE — Reproduce Training on Google Colab

Repo: `https://github.com/dodd-maindev/DRUG.git`  
Paper: *3DICE: interpretable 3D cross-modal learning for drug–target interaction prediction and large-scale drug discovery* (Bioinformatics, 2026)

> **Lưu ý quan trọng trước khi chạy:**  
> Repo Git chỉ chứa code và các file split CSV (`lists/db_new/`, `lists/KIBA/`), **không chứa** thư mục embedding (`embeddings/` và `drug/embeddings_atomic/`).  
> Bạn cần chọn **Cách A** (mount Google Drive chứa embedding đã precompute sẵn) hoặc **Cách B** (tự sinh embedding trên Colab ở Bước 3).

In [ ]:
# 1. Kiểm tra GPU & Mount Google Drive
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone repo và tạo các thư mục output bắt buộc
%cd /content
!rm -rf DRUG
!git clone https://github.com/dodd-maindev/DRUG.git
%cd /content/DRUG

# solver.py mở logs/training_log_*.csv trực tiếp mà chưa gọi makedirs
!mkdir -p logs saved best_models

In [ ]:
# 3. Cài đặt thư viện cần thiết để train (Colab đã có sẵn PyTorch + CUDA)
!pip install -q yacs torch_geometric scikit-learn pandas matplotlib tqdm

## Bước 4: Chuẩn bị Embeddings (Chọn Cách A hoặc Cách B)

In [ ]:
# === CÁCH A: Dùng embeddings đã precompute sẵn trên Google Drive ===
# Sửa lại đường dẫn Drive cho khớp với thư mục của bạn
DRIVE_PROT_EMB = "/content/drive/MyDrive/3DICE_data/embeddings"
DRIVE_DRUG_EMB = "/content/drive/MyDrive/3DICE_data/embeddings_atomic"

import os
if os.path.exists(DRIVE_PROT_EMB) and os.path.exists(DRIVE_DRUG_EMB):
    !ln -sfn "{DRIVE_PROT_EMB}" /content/DRUG/embeddings
    !ln -sfn "{DRIVE_DRUG_EMB}" /content/DRUG/drug/embeddings_atomic
    print("Đã link embeddings từ Google Drive!")
else:
    print("Chưa tìm thấy embeddings trên Drive. Hãy kiểm tra đường dẫn hoặc chạy Cách B bên dưới.")

In [ ]:
# === CÁCH B (Tùy chọn): Tự sinh embeddings nếu chưa có sẵn ===
# Chỉ chạy cell này nếu bạn chưa có embeddings trên Drive
# !pip install -q unimol_tools rdkit fair-esm biotite gemmi
# !mkdir -p /content/DRUG/drug/embeddings_atomic /content/DRUG/embeddings /content/DRUG/structures
#
# 1) Sinh Drug embeddings (Uni-Mol):
# !python drug/02_embed_drugs.py
#
# 2) Tải cấu trúc PDB/AlphaFoldDB và sinh Protein embeddings (ESM-IF1):
# !python protein/01_fetch_structures_v2.py --input_csv lists/db_new/db_train.csv --uniprot_col uniprot_id --out_dir structures --format mmcif --min_coverage 0.2 --alphafold_fallback 1
# !python protein/02_embed_protein_3D.py

## Bước 5: Kiểm tra nhanh DataLoader & Model trước khi train

In [ ]:
!python main.py
!python -c "from config.cfg import get_cfg_defaults; cfg = get_cfg_defaults(); print('Train:', cfg.DATA.TRAIN_CSV_PATH); print('Val:', cfg.DATA.VAL_CSV_PATH); print('Test:', cfg.DATA.TEST_CSV_PATH)"

## Bước 6: Chạy Training (Tái hiện kết quả)
- Theo bài báo (Mục 2.4): `BATCH_SIZE = 32`, `LR = 5e-5`, `DROPOUT = 0.3`, `WEIGHT_DECAY = 0`, `EPOCHS = 60` (trong `config/cfg.py` đang để mặc định `EPOCHS = 80`).
- Trên Colab nên đặt `NUM_WORKERS = 2` để tránh nghẽn RAM/CPU.

In [ ]:
# Chạy train trực tiếp với 60 epochs (khớp bài báo) và lưu log + best_models
!python -c "
import torch
from config.cfg import get_cfg_defaults
from model import Model
from solver import Solver

cfg = get_cfg_defaults()
cfg.SOLVER.EPOCHS = 60
cfg.DATA.NUM_WORKERS = 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Model(cfg)
solver = Solver(model, cfg, device=device, optim=torch.optim.Adam, loss_fn=cfg.SOLVER.LOSS_FN, eval=None)
solver.train(solver.train_dl, solver.val_dl)
"

In [ ]:
# 7. Backup kết quả (logs, best_models) về Google Drive sau khi train xong
!mkdir -p /content/drive/MyDrive/3DICE_outputs
!cp -r logs best_models /content/drive/MyDrive/3DICE_outputs/
print("Đã backup logs và best_models lên Google Drive!")